# Equipments Logic

This notebook builds AI logic for the Equipment module: inventory generation, live availability, task demand, equipment assignment, utilization scoring, predictive maintenance, conflict detection, auto-relocation, live feed events, and backend-ready payloads.

## 1. Setup

In [ ]:
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

now = datetime(2026, 4, 27, 9, 0)

## 2. Equipment Configuration

In [ ]:
EQUIPMENT_TYPES = {
    'Baggage Loaders': {'count': 22, 'task_type': 'baggage', 'service_rate': 22, 'maintenance_interval_hours': 180, 'critical_fuel_pct': 20},
    'Passenger Stairs': {'count': 12, 'task_type': 'boarding', 'service_rate': 180, 'maintenance_interval_hours': 240, 'critical_fuel_pct': 25},
    'Fuel Tractors': {'count': 14, 'task_type': 'fueling', 'service_rate': 1200, 'maintenance_interval_hours': 140, 'critical_fuel_pct': 30},
    'Tow Tractors': {'count': 16, 'task_type': 'pushback', 'service_rate': 1, 'maintenance_interval_hours': 160, 'critical_fuel_pct': 25},
    'Catering Vans': {'count': 10, 'task_type': 'catering', 'service_rate': 260, 'maintenance_interval_hours': 150, 'critical_fuel_pct': 25},
    'Water Truck': {'count': 8, 'task_type': 'water', 'service_rate': 500, 'maintenance_interval_hours': 130, 'critical_fuel_pct': 25},
    'Food Truck': {'count': 6, 'task_type': 'crew_food', 'service_rate': 120, 'maintenance_interval_hours': 170, 'critical_fuel_pct': 25},
}

ZONES = {
    'Zone A': {'x': 18, 'y': 24, 'near_gates': ['G01', 'G02', 'G03', 'G04']},
    'Zone B': {'x': 48, 'y': 16, 'near_gates': ['G05', 'G06', 'G07', 'G08']},
    'Zone C': {'x': 72, 'y': 30, 'near_gates': ['G09', 'G10', 'G11', 'G12']},
    'Depot': {'x': 32, 'y': 62, 'near_gates': []},
    'Fuel Pt': {'x': 62, 'y': 68, 'near_gates': ['G13', 'G14', 'G15']},
    'Gate B': {'x': 84, 'y': 54, 'near_gates': ['G16', 'G17', 'G18', 'G19', 'G20']},
}

STATUS_VALUES = ['available', 'in_use', 'maintenance', 'refueling', 'offline']
GATES = [f'G{str(i).zfill(2)}' for i in range(1, 25)]

## 3. Generate Equipment Inventory

In [ ]:
def generate_equipment_inventory():
    rows = []
    for equipment_type, cfg in EQUIPMENT_TYPES.items():
        prefix = ''.join(word[0] for word in equipment_type.split()).upper()
        for i in range(1, cfg['count'] + 1):
            age_years = round(random.uniform(0.5, 9.5), 1)
            hours_since_maintenance = round(random.uniform(5, cfg['maintenance_interval_hours'] * 1.25), 1)
            fuel_level = random.randint(8, 100)
            condition_score = np.clip(100 - age_years * 3.2 - (hours_since_maintenance / cfg['maintenance_interval_hours']) * 35 + np.random.normal(0, 6), 5, 100)
            status = random.choices(STATUS_VALUES, weights=[54, 25, 8, 8, 5])[0]
            if fuel_level <= cfg['critical_fuel_pct'] and status == 'available':
                status = 'refueling'
            if condition_score <= 35 and status in ['available', 'in_use']:
                status = 'maintenance'

            rows.append({
                'equipment_id': f'{prefix}-{i:03d}',
                'equipment_type': equipment_type,
                'task_type': cfg['task_type'],
                'zone': random.choice(list(ZONES.keys())),
                'current_gate': random.choice(GATES + ['Depot', 'Workshop']),
                'status': status,
                'age_years': age_years,
                'hours_since_maintenance': hours_since_maintenance,
                'maintenance_interval_hours': cfg['maintenance_interval_hours'],
                'fuel_level_pct': fuel_level,
                'battery_level_pct': random.randint(15, 100) if equipment_type in ['Baggage Loaders', 'Tow Tractors'] else np.nan,
                'condition_score': round(float(condition_score), 1),
                'service_rate': cfg['service_rate'],
                'active_task_id': None,
            })
    return pd.DataFrame(rows)


equipment_df = generate_equipment_inventory()
equipment_df.head(12)

## 4. Generate Flight Equipment Demand

In [ ]:
TASK_REQUIREMENTS = {
    'baggage': {'equipment_type': 'Baggage Loaders', 'duration': (18, 45), 'units': (1, 3)},
    'boarding': {'equipment_type': 'Passenger Stairs', 'duration': (15, 35), 'units': (1, 2)},
    'fueling': {'equipment_type': 'Fuel Tractors', 'duration': (20, 55), 'units': (1, 2)},
    'pushback': {'equipment_type': 'Tow Tractors', 'duration': (10, 25), 'units': (1, 1)},
    'catering': {'equipment_type': 'Catering Vans', 'duration': (20, 50), 'units': (1, 2)},
    'water': {'equipment_type': 'Water Truck', 'duration': (10, 28), 'units': (1, 1)},
    'crew_food': {'equipment_type': 'Food Truck', 'duration': (12, 30), 'units': (1, 1)},
}


def generate_equipment_tasks(n_flights=32):
    rows = []
    priority_rank_map = {'Normal': 0, 'VIP': 1, 'Connection Critical': 2, 'Emergency': 3}

    for flight_idx in range(n_flights):
        flight_id = f'{random.choice(["AI", "6E", "SG", "UK", "QP"])}{random.randint(100, 999)}'
        gate = random.choice(GATES)
        aircraft_size = random.choices(['Regional', 'Narrow Body', 'Wide Body'], weights=[15, 65, 20])[0]
        size_multiplier = {'Regional': 0.8, 'Narrow Body': 1.0, 'Wide Body': 1.8}[aircraft_size]
        base_time = now + timedelta(minutes=random.randint(-45, 180))
        priority = random.choices(['Normal', 'VIP', 'Emergency', 'Connection Critical'], weights=[76, 7, 3, 14])[0]

        for task_type, cfg in TASK_REQUIREMENTS.items():
            required_units = int(np.ceil(random.randint(*cfg['units']) * size_multiplier))
            duration = int(random.randint(*cfg['duration']) * size_multiplier)
            start = base_time + timedelta(minutes=random.randint(-10, 45))
            rows.append({
                'task_id': f'EQT-{flight_idx:03d}-{task_type.upper()[:4]}',
                'flight_id': flight_id,
                'gate': gate,
                'aircraft_size': aircraft_size,
                'task_type': task_type,
                'equipment_type': cfg['equipment_type'],
                'required_units': max(1, required_units),
                'task_start': start,
                'task_end': start + timedelta(minutes=duration),
                'duration_min': duration,
                'priority': priority,
                'priority_rank': priority_rank_map[priority],
            })
    return pd.DataFrame(rows).sort_values(['priority_rank', 'task_start'], ascending=[False, True]).reset_index(drop=True)


tasks_df = generate_equipment_tasks()
tasks_df.head(14)


## 5. Predictive Maintenance Score

In [ ]:
def maintenance_risk(row):
    interval_pressure = row['hours_since_maintenance'] / row['maintenance_interval_hours']
    age_pressure = row['age_years'] / 10
    condition_pressure = (100 - row['condition_score']) / 100
    fuel_pressure = max(0, 25 - row['fuel_level_pct']) / 25
    risk = 100 * (0.36 * interval_pressure + 0.22 * age_pressure + 0.34 * condition_pressure + 0.08 * fuel_pressure)
    return round(float(np.clip(risk, 0, 100)), 1)


def maintenance_severity(risk):
    if risk >= 72:
        return 'High'
    if risk >= 48:
        return 'Medium'
    return 'Low'


equipment_df['maintenance_risk_score'] = equipment_df.apply(maintenance_risk, axis=1)
equipment_df['maintenance_severity'] = equipment_df['maintenance_risk_score'].map(maintenance_severity)
equipment_df['battery_ready'] = equipment_df['battery_level_pct'].fillna(100) > 25
equipment_df['serviceable'] = (
    equipment_df['status'].isin(['available', 'in_use']) &
    (equipment_df['maintenance_risk_score'] < 72) &
    (equipment_df['fuel_level_pct'] > 15) &
    equipment_df['battery_ready']
)

equipment_df[['equipment_id', 'equipment_type', 'status', 'fuel_level_pct', 'condition_score', 'maintenance_risk_score', 'maintenance_severity', 'serviceable']].sort_values('maintenance_risk_score', ascending=False).head(15)


## 6. Equipment Assignment Engine

In [ ]:
def zone_distance(zone, gate):
    if gate in ZONES[zone]['near_gates']:
        return 0
    gate_num = int(gate.replace('G', '')) if gate.startswith('G') else 12
    zone_center = np.mean([int(g.replace('G', '')) for g in ZONES[zone]['near_gates']]) if ZONES[zone]['near_gates'] else 12
    return abs(gate_num - zone_center)


def assignment_score(equipment, task):
    distance_penalty = zone_distance(equipment['zone'], task['gate']) * 4
    maintenance_penalty = equipment['maintenance_risk_score'] * 0.55
    fuel_penalty = max(0, 45 - equipment['fuel_level_pct']) * 0.7
    priority_bonus = {'Normal': 0, 'VIP': -5, 'Emergency': -16, 'Connection Critical': -10}.get(task['priority'], 0)
    idle_bonus = -8 if equipment['status'] == 'available' else 0
    return distance_penalty + maintenance_penalty + fuel_penalty + priority_bonus + idle_bonus


def assign_equipment(equipment, tasks, max_tasks=90):
    pool = equipment[equipment['serviceable']].copy()
    schedules = {equipment_id: [] for equipment_id in pool['equipment_id']}
    assignment_load = {equipment_id: 0 for equipment_id in pool['equipment_id']}
    assignments = []
    selected_tasks = tasks.sort_values(['priority_rank', 'task_start', 'required_units'], ascending=[False, True, False]).head(max_tasks)

    def has_conflict(equipment_row, task):
        travel_buffer = max(6, int(zone_distance(equipment_row['zone'], task['gate']) * 2))
        return any(
            task['task_start'] < end + timedelta(minutes=travel_buffer) and start < task['task_end'] + timedelta(minutes=travel_buffer)
            for start, end in schedules[equipment_row['equipment_id']]
        )

    for _, task in selected_tasks.iterrows():
        candidates = pool[pool['equipment_type'] == task['equipment_type']].copy()
        if candidates.empty:
            assignments.append({
                'task_id': task['task_id'],
                'flight_id': task['flight_id'],
                'gate': task['gate'],
                'equipment_type': task['equipment_type'],
                'required_units': task['required_units'],
                'assigned_count': 0,
                'coverage_pct': 0.0,
                'assigned_equipment': '',
                'assignment_status': 'unfilled',
            })
            continue

        candidates = candidates[~candidates.apply(lambda row: has_conflict(row, task), axis=1)].copy()
        if candidates.empty:
            assignments.append({
                'task_id': task['task_id'],
                'flight_id': task['flight_id'],
                'gate': task['gate'],
                'equipment_type': task['equipment_type'],
                'required_units': task['required_units'],
                'assigned_count': 0,
                'coverage_pct': 0.0,
                'assigned_equipment': '',
                'assignment_status': 'unfilled',
            })
            continue

        candidates['score'] = candidates.apply(
            lambda row: assignment_score(row, task) + assignment_load[row['equipment_id']] * 7,
            axis=1,
        )
        chosen = candidates.sort_values(['score', 'fuel_level_pct', 'condition_score'], ascending=[True, False, False]).head(int(task['required_units']))
        for equipment_id in chosen['equipment_id']:
            schedules[equipment_id].append((task['task_start'], task['task_end']))
            assignment_load[equipment_id] += 1
        status = 'filled' if len(chosen) >= task['required_units'] else 'partial'
        assignments.append({
            'task_id': task['task_id'],
            'flight_id': task['flight_id'],
            'gate': task['gate'],
            'equipment_type': task['equipment_type'],
            'required_units': task['required_units'],
            'assigned_count': len(chosen),
            'coverage_pct': round(len(chosen) / max(task['required_units'], 1) * 100, 1),
            'assigned_equipment': ','.join(chosen['equipment_id'].tolist()),
            'assignment_status': status,
        })

    return pd.DataFrame(assignments)


assignments_df = assign_equipment(equipment_df, tasks_df)
assignments_df.head(20)


## 7. Utilization and Shortage Detection

In [ ]:
def calculate_equipment_capacity(equipment, tasks, horizon_minutes=90):
    horizon_end = now + timedelta(minutes=horizon_minutes)
    active_tasks = tasks[(tasks['task_start'] <= horizon_end) & (tasks['task_end'] >= now)].copy()
    demand = active_tasks.groupby('equipment_type')['required_units'].sum()
    total = equipment.groupby('equipment_type').size()
    serviceable = equipment[equipment['serviceable']].groupby('equipment_type').size()
    maintenance = equipment[equipment['status'] == 'maintenance'].groupby('equipment_type').size()
    low_fuel = equipment[equipment['fuel_level_pct'] <= 25].groupby('equipment_type').size()

    rows = []
    for equipment_type in EQUIPMENT_TYPES.keys():
        needed = int(demand.get(equipment_type, 0))
        usable = int(serviceable.get(equipment_type, 0))
        utilization = needed / max(usable, 1) * 100
        rows.append({
            'equipment_type': equipment_type,
            'total_units': int(total.get(equipment_type, 0)),
            'serviceable_units': usable,
            'maintenance_units': int(maintenance.get(equipment_type, 0)),
            'low_fuel_units': int(low_fuel.get(equipment_type, 0)),
            'demand_next_90m': needed,
            'capacity_gap': usable - needed,
            'utilization_pct': round(utilization, 1),
            'capacity_status': 'shortage' if needed > usable else 'tight' if utilization >= 85 else 'ok',
        })
    return pd.DataFrame(rows).sort_values(['capacity_status', 'utilization_pct'], ascending=[False, False]).reset_index(drop=True)


capacity_df = calculate_equipment_capacity(equipment_df, tasks_df)
capacity_df

## 8. Auto-Relocation Logic

In [ ]:
def zone_demand(tasks, horizon_minutes=90):
    horizon_end = now + timedelta(minutes=horizon_minutes)
    active = tasks[(tasks['task_start'] <= horizon_end) & (tasks['task_end'] >= now)].copy()
    def gate_to_zone(gate):
        for zone, cfg in ZONES.items():
            if gate in cfg['near_gates']:
                return zone
        return 'Depot'
    active['target_zone'] = active['gate'].map(gate_to_zone)
    return active.groupby('target_zone')['required_units'].sum().reindex(ZONES.keys(), fill_value=0)


def build_relocation_plan(equipment, tasks):
    demand = zone_demand(tasks)
    supply = equipment[equipment['serviceable']].groupby('zone').size().reindex(ZONES.keys(), fill_value=0)
    zone_table = pd.DataFrame({'zone': list(ZONES.keys()), 'serviceable_supply': supply.values, 'demand_next_90m': demand.values})
    zone_table['gap'] = zone_table['serviceable_supply'] - zone_table['demand_next_90m']
    zone_table['status'] = np.where(zone_table['gap'] < 0, 'busy', np.where(zone_table['gap'] > 5, 'idle', 'active'))

    shortage_zones = zone_table[zone_table['gap'] < 0].sort_values('gap')
    surplus_zones = zone_table[zone_table['gap'] > 2].sort_values('gap', ascending=False)
    moves = []
    for _, shortage in shortage_zones.iterrows():
        remaining_need = abs(int(shortage['gap']))
        for _, surplus in surplus_zones.iterrows():
            movable = max(0, int(surplus['gap']) - 2)
            if movable <= 0 or remaining_need <= 0:
                continue
            move_units = min(movable, remaining_need)
            moves.append({
                'from_zone': surplus['zone'],
                'to_zone': shortage['zone'],
                'units': move_units,
                'reason': f"Cover {shortage['zone']} equipment shortage for next 90 minutes",
            })
            remaining_need -= move_units
    return zone_table, pd.DataFrame(moves)


zone_status_df, relocation_df = build_relocation_plan(equipment_df, tasks_df)
display(zone_status_df)
relocation_df

## 9. Equipment Alerts and Recommendations

In [ ]:
def build_equipment_recommendations(equipment, capacity, assignments, relocation):
    alerts = []

    for _, row in capacity[capacity['capacity_status'] != 'ok'].iterrows():
        alerts.append({
            'priority_rank': 1 if row['capacity_status'] == 'shortage' else 2,
            'type': 'capacity_pressure',
            'asset': row['equipment_type'],
            'severity': 'High' if row['capacity_status'] == 'shortage' else 'Medium',
            'message': f"{row['equipment_type']} {row['capacity_status']}: demand {row['demand_next_90m']}, serviceable {row['serviceable_units']}.",
            'recommendation': 'Release reserve units or delay low-priority tasks.' if row['capacity_status'] == 'shortage' else 'Pre-position standby units near active gates.',
        })

    risky = equipment[equipment['maintenance_severity'] == 'High'].sort_values('maintenance_risk_score', ascending=False).head(12)
    for _, row in risky.iterrows():
        alerts.append({
            'priority_rank': 1,
            'type': 'predictive_maintenance',
            'asset': row['equipment_id'],
            'severity': 'High',
            'message': f"{row['equipment_id']} has maintenance risk {row['maintenance_risk_score']}.",
            'recommendation': 'Route to workshop after current task and assign backup unit.',
        })

    low_fuel = equipment[(equipment['fuel_level_pct'] <= 20) & (equipment['status'] != 'maintenance')].head(10)
    for _, row in low_fuel.iterrows():
        alerts.append({
            'priority_rank': 2,
            'type': 'low_fuel',
            'asset': row['equipment_id'],
            'severity': 'Medium',
            'message': f"{row['equipment_id']} fuel level is {row['fuel_level_pct']}%.",
            'recommendation': 'Send to Fuel Pt before assigning another task.',
        })

    gaps = assignments[assignments['assignment_status'].isin(['partial', 'unfilled'])].head(12)
    for _, row in gaps.iterrows():
        alerts.append({
            'priority_rank': 1 if row['assignment_status'] == 'unfilled' else 2,
            'type': 'assignment_gap',
            'asset': row['equipment_type'],
            'severity': 'High' if row['assignment_status'] == 'unfilled' else 'Medium',
            'message': f"{row['task_id']} for {row['flight_id']} is {row['assignment_status']}.",
            'recommendation': 'Reallocate nearest available unit and notify ground team.',
        })

    if not relocation.empty:
        for _, row in relocation.iterrows():
            alerts.append({
                'priority_rank': 2,
                'type': 'auto_relocation',
                'asset': f"{row['from_zone']} -> {row['to_zone']}",
                'severity': 'Medium',
                'message': f"Move {row['units']} units from {row['from_zone']} to {row['to_zone']}.",
                'recommendation': row['reason'],
            })

    return pd.DataFrame(alerts).sort_values(['priority_rank', 'severity']).reset_index(drop=True)


recommendations_df = build_equipment_recommendations(equipment_df, capacity_df, assignments_df, relocation_df)
recommendations_df.head(20)

## 10. Live Equipment Feed

In [ ]:
FEED_TEMPLATES = [
    '{asset} dispatched to {gate}',
    '{asset} returned to depot',
    '{asset} maintenance check started',
    '{asset} refueling complete',
    '{asset} repositioned to {zone}',
    '{asset} allocated to flight {flight}',
]


def generate_equipment_feed(equipment, assignments, n=30):
    feed = []
    assigned_assets = []
    for asset_list in assignments['assigned_equipment'].dropna():
        assigned_assets.extend([asset for asset in asset_list.split(',') if asset])
    candidate_assets = assigned_assets or equipment['equipment_id'].sample(min(n, len(equipment)), random_state=SEED).tolist()

    for i in range(n):
        asset = random.choice(candidate_assets)
        row = equipment[equipment['equipment_id'] == asset].iloc[0]
        template = random.choice(FEED_TEMPLATES)
        event_time = now - timedelta(minutes=random.randint(0, 90))
        feed.append({
            'event_id': f'EQFEED-{1000 + i}',
            'time': event_time.strftime('%H:%M:%S'),
            'equipment_id': asset,
            'equipment_type': row['equipment_type'],
            'zone': row['zone'],
            'message': template.format(asset=asset, gate=random.choice(GATES), zone=random.choice(list(ZONES.keys())), flight=random.choice(tasks_df['flight_id'].tolist())),
            'severity': 'High' if row['maintenance_severity'] == 'High' else 'Normal',
        })
    return pd.DataFrame(feed).sort_values('time', ascending=False).reset_index(drop=True)


feed_df = generate_equipment_feed(equipment_df, assignments_df)
feed_df.head(15)

## 11. Equipment KPI Snapshot

In [ ]:
def build_equipment_kpis(equipment, capacity, assignments, recommendations):
    total_units = len(equipment)
    serviceable_units = int(equipment['serviceable'].sum())
    in_use = int((equipment['status'] == 'in_use').sum())
    maintenance = int((equipment['status'] == 'maintenance').sum())
    low_fuel = int((equipment['fuel_level_pct'] <= 25).sum())
    high_maintenance = int((equipment['maintenance_severity'] == 'High').sum())
    fill_rate = (assignments['assignment_status'] == 'filled').mean() * 100 if len(assignments) else 0
    avg_util = capacity['utilization_pct'].replace(999, 100).mean()
    shortage_types = int((capacity['capacity_status'] == 'shortage').sum())
    health = equipment['condition_score'].mean()
    operation_score = 0.30 * (serviceable_units / total_units * 100) + 0.25 * fill_rate + 0.20 * health + 0.25 * max(0, 100 - min(avg_util, 140) * 0.55)
    operation_score -= min(shortage_types * 6, 24) + min(high_maintenance * 0.8, 18)

    return pd.DataFrame([{
        'total_units': total_units,
        'serviceable_units': serviceable_units,
        'in_use_units': in_use,
        'maintenance_units': maintenance,
        'low_fuel_units': low_fuel,
        'high_maintenance_risk_units': high_maintenance,
        'assignment_fill_rate_pct': round(fill_rate, 1),
        'avg_utilization_pressure_pct': round(avg_util, 1),
        'shortage_equipment_types': shortage_types,
        'critical_alerts': int((recommendations['priority_rank'] == 1).sum()) if len(recommendations) else 0,
        'equipment_operation_score': round(float(np.clip(operation_score, 0, 100)), 1),
    }])


kpi_df = build_equipment_kpis(equipment_df, capacity_df, assignments_df, recommendations_df)
kpi_df

## 12. Equipment Visuals

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

type_counts = equipment_df['equipment_type'].value_counts().reindex(EQUIPMENT_TYPES.keys(), fill_value=0)
axes[0, 0].barh(type_counts.index, type_counts.values, color='#1565c0')
axes[0, 0].set_title('Inventory by Equipment Type')
axes[0, 0].set_xlabel('Units')

status_counts = equipment_df['status'].value_counts().reindex(STATUS_VALUES, fill_value=0)
axes[0, 1].bar(status_counts.index, status_counts.values, color='#00897b')
axes[0, 1].set_title('Equipment Status')
axes[0, 1].tick_params(axis='x', rotation=35)

risk_counts = equipment_df['maintenance_severity'].value_counts().reindex(['Low', 'Medium', 'High'], fill_value=0)
axes[1, 0].bar(risk_counts.index, risk_counts.values, color=['#2e7d32', '#f9a825', '#c62828'])
axes[1, 0].set_title('Predictive Maintenance Risk')

axes[1, 1].barh(capacity_df['equipment_type'], capacity_df['utilization_pct'], color='#6a1b9a')
axes[1, 1].axvline(85, color='#f9a825', linestyle='--', label='Tight')
axes[1, 1].axvline(100, color='#c62828', linestyle='--', label='Shortage')
axes[1, 1].set_title('Utilization Pressure')
axes[1, 1].set_xlabel('Utilization %')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 13. Backend-Ready Payload

In [ ]:
def build_equipment_payload(equipment, capacity, assignments, recommendations, zone_status, relocation, feed, kpis):
    inventory_columns = [
        'equipment_id', 'equipment_type', 'zone', 'current_gate', 'status', 'fuel_level_pct',
        'battery_level_pct', 'condition_score', 'maintenance_risk_score', 'maintenance_severity', 'serviceable'
    ]
    return {
        'generated_at': now.isoformat(),
        'kpis': kpis.iloc[0].to_dict(),
        'inventory': equipment[inventory_columns].to_dict(orient='records'),
        'capacity': capacity.to_dict(orient='records'),
        'assignments': assignments.head(50).to_dict(orient='records'),
        'recommendations': recommendations.head(20).to_dict(orient='records'),
        'zone_status': zone_status.to_dict(orient='records'),
        'relocation_plan': relocation.to_dict(orient='records') if not relocation.empty else [],
        'live_feed': feed.head(20).to_dict(orient='records'),
        'maintenance_watchlist': equipment.sort_values('maintenance_risk_score', ascending=False).head(15)[inventory_columns].to_dict(orient='records'),
    }


payload = build_equipment_payload(equipment_df, capacity_df, assignments_df, recommendations_df, zone_status_df, relocation_df, feed_df, kpi_df)
payload

In [ ]:
print('EQUIPMENT SUMMARY')
print('=================')
print(f"Total units: {payload['kpis']['total_units']}")
print(f"Serviceable units: {payload['kpis']['serviceable_units']}")
print(f"Maintenance units: {payload['kpis']['maintenance_units']}")
print(f"Low fuel units: {payload['kpis']['low_fuel_units']}")
print(f"Assignment fill rate: {payload['kpis']['assignment_fill_rate_pct']}%")
print(f"Shortage equipment types: {payload['kpis']['shortage_equipment_types']}")
print(f"Equipment Operation Score: {payload['kpis']['equipment_operation_score']}")
print('\nTop equipment recommendations:')
for item in payload['recommendations'][:5]:
    print(f"- [{item['type']}] {item['message']} {item['recommendation']}")